<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Medium/TensorFlow/Tf_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Libraries

In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import Model, Sequential, layers, optimizers, losses, metrics
from keras.datasets import mnist
from keras.layers import Dense, GlobalAveragePooling2D

### Dataset Preparation

In [2]:
!pip install -q kaggle

In [3]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"hbahruz","key":"4f925bc35f1b4f8677fc6f8061cceb06"}'}

In [4]:
!kaggle datasets download -d akhiljethwa/forest-vs-desert

Dataset URL: https://www.kaggle.com/datasets/akhiljethwa/forest-vs-desert
License(s): CC-BY-NC-SA-4.0
 66% 5.00M/7.54M [00:00<00:00, 45.9MB/s]
100% 7.54M/7.54M [00:00<00:00, 63.9MB/s]


In [5]:
import zipfile

crab_species_zip = 'forest-vs-desert.zip'

def extract_zip(file_path, extract_to='.'):
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

extract_zip(crab_species_zip, 'forest_vs_desert')

In [19]:
batch_size = 8
img_height = 224
img_width = 224

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'forest_vs_desert/Data',
    validation_split=0.2,  # Use 20% of training data for validation
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'forest_vs_desert/Data',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=1024)
val_ds = val_ds.cache().prefetch(buffer_size=1024)

Found 802 files belonging to 2 classes.
Using 642 files for training.
Found 802 files belonging to 2 classes.
Using 160 files for validation.


### Transfer Learning 1 - Training FC layers by freezing the rest

In [20]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    strategy = tf.distribute.MirroredStrategy()
    print(f"Number of GPUs: {len(gpus)}")
else:
    strategy = tf.distribute.get_strategy()
    print("No GPUs available, using default strategy")

Number of GPUs: 1


In [24]:
with strategy.scope():
    # Search: https://www.tensorflow.org/api_docs/python/tf/keras/applications/ResNet50
    # include_top = True adds FC layers to the end
    # include_top = False removes FC layers at the end of the architecture
    resnet50 = tf.keras.applications.ResNet50(
        include_top=True,
        weights='imagenet',
        input_shape=(img_height, img_width, 3)
    )

    # Freezing all layers
    for layer in resnet50.layers:
        layer.trainable = False

    resnet50.layers.pop() # Remove the last FC layer

    # Add FC layer to classify 2 labels
    x = resnet50.layers[-1].output
    predictions = Dense(2, activation = "softmax")(x)

    model = Model(inputs = resnet50.input, outputs = predictions)
    optimizer = optimizers.Adam(learning_rate=0.01)
    loss_fn = losses.SparseCategoricalCrossentropy()
    metric = metrics.SparseCategoricalAccuracy()

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=[metric])

102967424/102967424 [==============================] - 1s 0us/step


In [25]:
resnet50.summary()

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 230, 230, 3)          0         ['input_3[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 112, 112, 64)         9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1_conv[0][0]']          
 on)                                                                                       

In [26]:
model.fit(train_ds, validation_data = val_ds, epochs=8)

Epoch 1/8
81/81 [==============================] - 12s 67ms/step - loss: 0.5956 - sparse_categorical_accuracy: 0.8879 - val_loss: 0.5125 - val_sparse_categorical_accuracy: 0.9187
Epoch 2/8
81/81 [==============================] - 3s 38ms/step - loss: 0.4610 - sparse_categorical_accuracy: 0.9470 - val_loss: 0.4052 - val_sparse_categorical_accuracy: 0.9438
Epoch 3/8
81/81 [==============================] - 3s 39ms/step - loss: 0.3739 - sparse_categorical_accuracy: 0.9579 - val_loss: 0.3444 - val_sparse_categorical_accuracy: 0.9312
Epoch 4/8
81/81 [==============================] - 3s 41ms/step - loss: 0.3148 - sparse_categorical_accuracy: 0.9579 - val_loss: 0.2997 - val_sparse_categorical_accuracy: 0.9438
Epoch 5/8
81/81 [==============================] - 3s 39ms/step - loss: 0.2729 - sparse_categorical_accuracy: 0.9579 - val_loss: 0.2704 - val_sparse_categorical_accuracy: 0.9438
Epoch 6/8
81/81 [==============================] - 3s 39ms/step - loss: 0.2415 - sparse_categorical_accuracy:

### Transfer Learning 2 - Fine Tuning

In [28]:
with strategy.scope():
    fine_tune = tf.keras.applications.ResNet50(
        include_top=True,
        weights='imagenet',
        input_shape=(img_height, img_width, 3)
    )

    # Freezing all layers
    for layer in fine_tune.layers:
        layer.trainable = False

    fine_tune.layers.pop() # Remove the last FC layer

    # Add FC layer to classify 2 labels
    x = fine_tune.layers[-1].output
    predictions = Dense(2, activation = "softmax")(x)

    model = Model(inputs = fine_tune.input, outputs = predictions)
    optimizer = optimizers.Adam(learning_rate=0.01)
    loss_fn = losses.SparseCategoricalCrossentropy()
    metric = metrics.SparseCategoricalAccuracy()

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=[metric])

In [30]:
model.fit(train_ds, validation_data = val_ds, epochs=8)

Epoch 1/8
81/81 [==============================] - 4s 44ms/step - loss: 0.2410 - sparse_categorical_accuracy: 0.9626 - val_loss: 0.2484 - val_sparse_categorical_accuracy: 0.9312
Epoch 2/8
81/81 [==============================] - 3s 40ms/step - loss: 0.2169 - sparse_categorical_accuracy: 0.9626 - val_loss: 0.2329 - val_sparse_categorical_accuracy: 0.9312
Epoch 3/8
81/81 [==============================] - 3s 39ms/step - loss: 0.1979 - sparse_categorical_accuracy: 0.9642 - val_loss: 0.2218 - val_sparse_categorical_accuracy: 0.9312
Epoch 4/8
81/81 [==============================] - 3s 39ms/step - loss: 0.1825 - sparse_categorical_accuracy: 0.9657 - val_loss: 0.2149 - val_sparse_categorical_accuracy: 0.9312
Epoch 5/8
81/81 [==============================] - 3s 42ms/step - loss: 0.1696 - sparse_categorical_accuracy: 0.9657 - val_loss: 0.2089 - val_sparse_categorical_accuracy: 0.9312
Epoch 6/8
81/81 [==============================] - 3s 40ms/step - loss: 0.1589 - sparse_categorical_accuracy: 